# 02. Write와 Select

Write는 정보를 외부 기록에 남기는 단계이고, Select는 현재 질문에 필요한 기록만 고르는 단계입니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "openai==3.8.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

각 노트북은 빈 커널에서 시작할 수 있도록 연결 코드를 포함합니다.

In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
client = OpenAI()
MODEL = "gpt-5.4-mini"
print("API 실습 준비 완료")

API 실습 준비 완료


## 2. 기록을 JSON으로 저장하고 다시 읽습니다

예제 데이터는 짧게 인라인으로 두고, 임시 폴더에 저장하여 작업 폴더에 의존하지 않게 합니다.

In [3]:
import json
from pathlib import Path
from tempfile import gettempdir

records = [
    {"id": "E01-p2", "topic": "휴관", "text": "매주 월요일 휴관"},
    {"id": "E01-p3", "topic": "대출", "text": "7권, 14일 대출"},
    {"id": "E01-p5", "topic": "주차", "text": "90분 무료"},
]
memory_path = Path(gettempdir()) / "ch08-memory.json"
memory_path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
loaded = json.loads(memory_path.read_text(encoding="utf-8"))
print("다시 읽은 기록:", len(loaded))

다시 읽은 기록: 3


## 3. 질문에 필요한 기록만 선택합니다

여기서는 학생이 식별자를 직접 지정합니다. 자동 검색은 Ch 9에서 다룹니다.

In [4]:
selected_ids = {"E01-p3", "E01-p5"}
selected = [row for row in loaded if row["id"] in selected_ids]
for row in selected:
    print(row["id"], row["topic"], row["text"])

E01-p3 대출 7권, 14일 대출
E01-p5 주차 90분 무료


## 4. 선택한 기록으로 답을 생성합니다

모델에는 저장소 전체 대신 선택한 두 기록만 전달합니다.

In [5]:
context = "\n".join(f"[{r['id']}] {r['text']}" for r in selected)
prompt = f'근거:\n{context}\n\n질문: 대출 권수와 주차 무료 시간을 알려 주세요.'

In [6]:
response = client.responses.create(
    model=MODEL,
    instructions="제공된 근거만 사용하고 문장마다 식별자를 붙이세요.",
    input=prompt,
    max_output_tokens=300,
    reasoning={"effort": "none"},
)
print(response.output_text)
print("input_tokens:", response.usage.input_tokens)
print("output_tokens:", response.usage.output_tokens)

대출은 7권이며, 대출 기간은 14일입니다. [E01-p3]  
주차는 90분 무료입니다. [E01-p5]
input_tokens: 71
output_tokens: 43


## 5. 일부러 실패해 보기

저장만 하고 선택한 기록을 입력에 넣지 않으면 근거가 비어 있습니다. 호출 전에 검사하여 실패를 포착합니다.

In [7]:
def require_context(rows: list[dict[str, str]]) -> None:
    if not rows:
        raise ValueError("선택한 근거가 없습니다.")

try:
    require_context([])
except ValueError as error:
    print("예상한 실패:", error)
else:
    raise AssertionError("빈 근거 오류가 발생해야 합니다.")

예상한 실패: 선택한 근거가 없습니다.


## 정리

외부 기록은 저장만으로 모델에 전달되지 않습니다. 질문에 맞는 항목을 선택하여 호출 입력에 넣어야 합니다.